In [1]:
import numpy as np
import gymnasium as gym

# ----------------------------
# Part 1: Simple Hand-Coded MDP
# ----------------------------

# Define transition probabilities: P[state][action] = (prob, next_state, reward, done)
P = {
    0: {
        0: [(1.0, 0, 0, False)],
        1: [(1.0, 1, 0, False)]
    },
    1: {
        0: [(1.0, 0, 0, False)],
        1: [(1.0, 2, 1, False)]
    },
    2: {
        0: [(1.0, 1, 0, False)],
        1: [(1.0, 3, 10, True)]  # Goal state
    },
    3: {
        0: [(1.0, 3, 0, True)],
        1: [(1.0, 3, 0, True)]
    }
}

n_states = 4
n_actions = 2
gamma = 0.9

# Value Iteration for Hand-coded MDP
def value_iteration(P, n_states, n_actions, gamma=0.9, theta=1e-6):
    V = np.zeros(n_states)
    while True:
        delta = 0
        for s in range(n_states):
            v = V[s]
            action_values = []
            for a in range(n_actions):
                q = 0
                for prob, next_state, reward, done in P[s][a]:
                    q += prob * (reward + gamma * V[next_state] * (not done))
                action_values.append(q)
            V[s] = max(action_values)
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break

    # Extract policy
    policy = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        action_values = []
        for a in range(n_actions):
            q = 0
            for prob, next_state, reward, done in P[s][a]:
                q += prob * (reward + gamma * V[next_state] * (not done))
            action_values.append(q)
        policy[s] = np.argmax(action_values)
    return policy, V

# Policy Iteration for Hand-coded MDP
def policy_iteration(P, n_states, n_actions, gamma=0.9):
    policy = np.zeros(n_states, dtype=int)
    V = np.zeros(n_states)

    while True:
        # Policy Evaluation
        while True:
            delta = 0
            for s in range(n_states):
                v = V[s]
                a = policy[s]
                V[s] = sum(prob * (reward + gamma * V[next_state] * (not done))
                           for prob, next_state, reward, done in P[s][a])
                delta = max(delta, abs(v - V[s]))
            if delta < 1e-6:
                break

        # Policy Improvement
        stable = True
        for s in range(n_states):
            old_action = policy[s]
            action_values = []
            for a in range(n_actions):
                q = 0
                for prob, next_state, reward, done in P[s][a]:
                    q += prob * (reward + gamma * V[next_state] * (not done))
                action_values.append(q)
            policy[s] = np.argmax(action_values)
            if old_action != policy[s]:
                stable = False
        if stable:
            break
    return policy, V

print("--- Hand-coded MDP ---")
vi_policy, vi_value = value_iteration(P, n_states, n_actions, gamma)
pi_policy, pi_value = policy_iteration(P, n_states, n_actions, gamma)

print("Value Iteration Policy:", vi_policy)
print("Value Iteration Values:", vi_value)
print("Policy Iteration Policy:", pi_policy)
print("Policy Iteration Values:", pi_value)


# ----------------------------
# Part 2: OpenAI Gym FrozenLake
# ----------------------------

env = gym.make("FrozenLake-v1", is_slippery=False)  # Deterministic version

n_states = env.observation_space.n
n_actions = env.action_space.n
P = env.unwrapped.P # Access P through unwrapped
gamma = 0.99

# Value Iteration for FrozenLake
def value_iteration_frozen(P, n_states, n_actions, gamma=0.99, theta=1e-6):
    V = np.zeros(n_states)
    while True:
        delta = 0
        for s in range(n_states):
            v = V[s]
            V[s] = max(sum(p * (r + gamma * V[s_]) for p, s_, r, _ in P[s][a]) for a in range(n_actions))
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break

    policy = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        Q = np.zeros(n_actions)
        for a in range(n_actions):
            Q[a] = sum(p * (r + gamma * V[s_]) for p, s_, r, _ in P[s][a])
        policy[s] = np.argmax(Q)
    return policy, V

# Policy Iteration for FrozenLake
def policy_iteration_frozen(P, n_states, n_actions, gamma=0.99, max_iter=1000):
    policy = np.zeros(n_states, dtype=int)
    V = np.zeros(n_states)

    def policy_evaluation(policy, V):
        while True:
            delta = 0
            for s in range(n_states):
                v = V[s]
                a = policy[s]
                V[s] = sum(p * (r + gamma * V[s_]) for p, s_, r, _ in P[s][a])
                delta = max(delta, abs(v - V[s]))
            if delta < 1e-6:
                break
        return V

    for _ in range(max_iter):
        V = policy_evaluation(policy, V)
        stable = True
        for s in range(n_states):
            old_action = policy[s]
            Q = np.zeros(n_actions)
            for a in range(n_actions):
                Q[a] = sum(p * (r + gamma * V[s_]) for p, s_, r, _ in P[s][a])
            policy[s] = np.argmax(Q)
            if old_action != policy[s]:
                stable = False
        if stable:
            break
    return policy, V

print("\n--- FrozenLake-v1 ---")
vi_policy, vi_value = value_iteration_frozen(P, n_states, n_actions)
pi_policy, pi_value = policy_iteration_frozen(P, n_states, n_actions)

# Helper: Render policy with arrows
action_symbols = {0: '←', 1: '↓', 2: '→', 3: '↑'}
def render_policy(policy, shape=(4,4)):
    return np.array([action_symbols[a] for a in policy]).reshape(shape)

print("Value Iteration Policy:")
print(render_policy(vi_policy))
print("Policy Iteration Policy:")
print(render_policy(pi_policy))

# Test rollout function
def run_policy(env, policy, episodes=3):
    for ep in range(episodes):
        obs, _ = env.reset()
        done, truncated = False, False
        steps = 0
        while not (done or truncated):
            action = policy[obs]
            obs, reward, done, truncated, _ = env.step(action)
            steps += 1
        print(f"Episode {ep+1}: finished in {steps} steps with reward {reward}")

env = gym.make("FrozenLake-v1", render_mode="ansi", is_slippery=False)
run_policy(env, vi_policy)

--- Hand-coded MDP ---
Value Iteration Policy: [1 1 1 0]
Value Iteration Values: [ 9. 10. 10.  0.]
Policy Iteration Policy: [1 1 1 0]
Policy Iteration Values: [ 9. 10. 10.  0.]

--- FrozenLake-v1 ---
Value Iteration Policy:
[['↓' '→' '↓' '←']
 ['↓' '←' '↓' '←']
 ['→' '↓' '↓' '←']
 ['←' '→' '→' '←']]
Policy Iteration Policy:
[['↓' '→' '↓' '←']
 ['↓' '←' '↓' '←']
 ['→' '↓' '↓' '←']
 ['←' '→' '→' '←']]
Episode 1: finished in 6 steps with reward 1.0
Episode 2: finished in 6 steps with reward 1.0
Episode 3: finished in 6 steps with reward 1.0


In [2]:
import numpy as np

# Grid size
rows, cols = 3, 4
gamma = 0.9
theta = 1e-6

# Rewards
goal = (2, 3)
trap = (1, 2)

def reward(state):
    if state == goal:
        return 15
    elif state == trap:
        return -10
    else:
        return -1

# Actions (Up, Down, Left, Right)
actions = [(-1,0), (1,0), (0,-1), (0,1)]
action_symbols = ['↑','↓','←','→']

def next_state(s, a):
    if s == goal or s == trap:
        return s
    x,y = s
    dx,dy = a
    nx, ny = x+dx, y+dy
    if 0 <= nx < rows and 0 <= ny < cols:
        return (nx, ny)
    return (x,y)

# Initialize values
V = np.zeros((rows, cols))

# Value Iteration
while True:
    delta = 0
    for i in range(rows):
        for j in range(cols):
            s = (i,j)
            v = V[i,j]
            if s == goal or s == trap:
                V[i,j] = reward(s)
            else:
                V[i,j] = max(
                    reward(s) + gamma * V[next_state(s,a)]
                    for a in actions
                )
            delta = max(delta, abs(v - V[i,j]))
    if delta < theta:
        break

# Extract policy
policy = np.full((rows, cols), ' ')
for i in range(rows):
    for j in range(cols):
        s = (i,j)
        if s == goal:
            policy[i,j] = 'G'
        elif s == trap:
            policy[i,j] = 'T'
        else:
            q_values = []
            for a_idx, a in enumerate(actions):
                s_next = next_state(s,a)
                q = reward(s) + gamma * V[s_next]
                q_values.append(q)
            policy[i,j] = action_symbols[np.argmax(q_values)]

print("Optimal Value Function:")
print(np.round(V,2))
print("\nOptimal Policy:")
print(policy)


Optimal Value Function:
[[  4.76   6.4    8.22  10.25]
 [  6.4    8.22 -10.    12.5 ]
 [  8.22  10.25  12.5   15.  ]]

Optimal Policy:
[['↓' '↓' '→' '↓']
 ['↓' '↓' 'T' '↓']
 ['→' '→' '→' 'G']]
